# M0 model-family tuning with expanding time-aware cross-validation

This notebook selects the no-weather M0 classifier using model-development rows in the frozen Notebook 04 split, excluding every household assigned to Notebook 04B's intentional unseen-household holdout. Holdout labels therefore cannot influence model-family or hyperparameter selection. It does not load outer-test labels, fit a final model on test rows, or report outer-test metrics. P95 is primary and P90 is a sensitivity target; both use the same row IDs and chronological folds.

Logistic Regression, Random Forest, XGBoost, and CatBoost are compared with fixed expanding-window folds over the non-held-out selection pool. Primary selection metric is PR-AUC; ROC-AUC, Brier score, and fold-to-fold PR-AUC variability are secondary. Two non-ML baselines are included. The selected family, parameters, and feature list are saved for Notebook 06 and Notebook 10.


## Stage 0 — imports, configuration, and feature registry

The randomized search budgets below keep this local run executable while sampling the planned parameter spaces. Increase the budgets for a wider search when compute time permits; all families and targets share the same three folds.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import itertools
import json
import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
if not (ROOT / "data").is_dir() or not (ROOT / "notebooks").is_dir():
    raise RuntimeError(f"Could not identify project root: {ROOT}")
DATA_DIR = ROOT / "data" / "processed"
TABLE_DIR = ROOT / "reports" / "tables" / "05_model_training_tuning"
FIGURE_DIR = ROOT / "reports" / "figures" / "05_model_training_tuning"
for folder in [TABLE_DIR, FIGURE_DIR]: folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 20261007
FOLD_CUMULATIVE_ROW_FRACTIONS = [0.55, 0.70, 0.85, 1.0]
SELECTION_TOLERANCE_PR_AUC = 0.005
SEARCH_BUDGETS = {"Random Forest": 4, "XGBoost": 4, "CatBoost": 3}
N_JOBS = 2
CHUNK_SIZE = 100_000

COMMON_FEATURES = [
    "lag1_peak_kw", "lag2_peak_kw", "peak_d7", "peak_d14", "peak_d21", "peak_d28",
    "lag1_mean_kw", "lag7_mean_kw", "lag1_import_active_ratio", "lag7_import_active_ratio",
    "lag1_load_factor", "lag1_interval_p95_kw", "lag1_peak_hour_sin", "lag1_peak_hour_cos",
    "same_weekday_peak_mean4", "same_weekday_valid_count_4", "peak_mean_3d", "peak_mean_7d",
    "peak_mean_30d", "peak_max_7d", "peak_max_30d", "peak_std_7d", "peak_std_30d",
    "valid_peak_days_7d", "valid_peak_days_30d", "has_lag7", "peak_change_1d",
    "peak_mean3_over_30", "peak_mean_7d_over_30d", "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos", "is_weekend", "is_public_holiday", "is_poya_day",
]
P95_FEATURES = COMMON_FEATURES + [
    "lag1_peak_over_p95", "lag1_peak_minus_p95", "lag2_peak_over_p95", "lag7_peak_over_p95",
    "peak_mean_7d_over_p95", "p95_high_count_7d", "p95_high_count_30d",
    "p95_eligible_event_days_7d", "p95_eligible_event_days_30d", "p95_high_rate_30d",
    "p95_days_since_last_high",
]
P90_FEATURES = COMMON_FEATURES + [
    "lag1_peak_over_p90", "lag1_peak_minus_p90", "lag2_peak_over_p90", "lag7_peak_over_p90",
    "peak_mean_7d_over_p90", "p90_high_count_7d", "p90_high_count_30d",
    "p90_eligible_event_days_7d", "p90_eligible_event_days_30d", "p90_high_rate_30d",
    "p90_days_since_last_high",
]
FEATURES = {"P95": P95_FEATURES, "P90": P90_FEATURES}
DATASETS = {
    "P95": DATA_DIR / "05_model_ready_m0_p95.csv",
    "P90": DATA_DIR / "05_model_ready_m0_p90.csv",
}
CONTRACTS = {
    "P95": DATA_DIR / "05_model_ready_m0_p95_dtype_contract.json",
    "P90": DATA_DIR / "05_model_ready_m0_p90_dtype_contract.json",
}
SPLIT_PATH = DATA_DIR / "04_master_split.csv"
SPLIT_CONTRACT_PATH = DATA_DIR / "04_master_split_dtype_contract.json"
SPLIT_MANIFEST_PATH = ROOT / "reports" / "tables" / "04_split_design" / "04_split_design_manifest.json"
HOLDOUT_ASSIGNMENT_PATH = DATA_DIR / "04b_household_holdout_assignment.csv"
HOLDOUT_ASSIGNMENT_CONTRACT_PATH = DATA_DIR / "04b_household_holdout_assignment_dtype_contract.json"
HOLDOUT_MANIFEST_PATH = ROOT / "reports" / "tables" / "04b_unseen_household_holdout_design" / "04b_unseen_household_holdout_manifest.json"

print("Project:", ROOT)
print("Search budgets (randomized configs):", SEARCH_BUDGETS)
print("CV cumulative row boundaries:", FOLD_CUMULATIVE_ROW_FRACTIONS)
print("Outer-test target labels are excluded from this notebook's model datasets and metrics.")


## Stage 1 — verify the frozen split and load model-development rows only

The split and frozen 04B assignment manifests verify upstream hashes. This cell removes intentional-holdout household IDs before it scans model-development row IDs, then reads only the remaining physical rows; held-out labels and outer-test labels are not loaded for model selection.


In [ ]:
def read_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))
def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size): digest.update(block)
    return digest.hexdigest()

def read_contract(path):
    contract = read_json(path)
    return contract

def read_csv_with_contract(path, contract, usecols=None, nrows=None, skiprows=None):
    header = pd.read_csv(path, nrows=0, encoding=contract["encoding"], sep=contract["separator"]).columns.tolist()
    if header != list(contract["columns"]): raise RuntimeError(f"CSV header differs from its dtype contract: {path.name}")
    columns = list(contract["columns"]) if usecols is None else list(usecols)
    dates = [name for name in contract.get("date_columns", []) if name in columns]
    dtypes = {name: value for name, value in contract["columns"].items() if name in columns and value != "datetime"}
    reader = contract["reader"]
    return pd.read_csv(path, usecols=columns, dtype=dtypes or None, parse_dates=dates or None, nrows=nrows, skiprows=skiprows,
        true_values=reader["true_values"], false_values=reader["false_values"],
        keep_default_na=reader["keep_default_na"], na_values=reader["na_values"],
        encoding=contract["encoding"], sep=contract["separator"], low_memory=False)

def read_model_development_rows(path, contract, usecols, expected_ids):
    """Scan identifiers only, then parse only model-development records from the CSV."""
    selected_line_numbers=set()
    found_ids=set()
    physical_line=1  # Header is line zero for pandas' skiprows callable.
    for chunk in pd.read_csv(path,usecols=["row_id"],dtype={"row_id":"string"},chunksize=CHUNK_SIZE):
        ids=chunk["row_id"].astype(str).to_numpy()
        for offset,row_id in enumerate(ids):
            if row_id in expected_ids:
                selected_line_numbers.add(physical_line+offset)
                found_ids.add(row_id)
        physical_line+=len(chunk)
    if found_ids != expected_ids:
        missing_ids=sorted(expected_ids-found_ids)[:10]
        unexpected=sorted(found_ids-expected_ids)[:10]
        raise RuntimeError(f"Could not map frozen model-development IDs to {path.name}; missing={missing_ids}, unexpected={unexpected}")
    selected_count=len(selected_line_numbers)
    frame=read_csv_with_contract(path,contract,usecols=usecols,
        skiprows=lambda line_number: line_number > 0 and line_number not in selected_line_numbers)
    if len(frame)!=selected_count:
        raise RuntimeError(f"Selective model-development read returned {len(frame)} rows, expected {selected_count}")
    return frame, selected_count

required = [SPLIT_PATH, SPLIT_CONTRACT_PATH, SPLIT_MANIFEST_PATH, HOLDOUT_ASSIGNMENT_PATH, HOLDOUT_ASSIGNMENT_CONTRACT_PATH, HOLDOUT_MANIFEST_PATH, *DATASETS.values(), *CONTRACTS.values()]
missing = [str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError(f"Model-tuning inputs are missing: {missing}")
split_manifest = read_json(SPLIT_MANIFEST_PATH)
master_signature = split_manifest["master_split"]
if sha256(SPLIT_PATH) != master_signature["sha256"]:
    raise RuntimeError("Notebook 04 master split does not match its frozen manifest")
master_contract = read_contract(SPLIT_CONTRACT_PATH)
master = read_csv_with_contract(SPLIT_PATH, master_contract)
holdout_manifest = read_json(HOLDOUT_MANIFEST_PATH)
if not holdout_manifest.get("validation", {}).get("assignment_groups_are_disjoint_and_complete", False):
    raise RuntimeError("Notebook 04B household assignment did not pass validation")
if holdout_manifest.get("future_target_outcomes_used_for_assignment", True):
    raise RuntimeError("Notebook 04B assignment must not use future target outcomes")
if sha256(HOLDOUT_ASSIGNMENT_PATH) != holdout_manifest["assignment"]["sha256"]:
    raise RuntimeError("Notebook 04B household assignment does not match its frozen manifest")
assignment_contract = read_contract(HOLDOUT_ASSIGNMENT_CONTRACT_PATH)
holdout_assignment = read_csv_with_contract(HOLDOUT_ASSIGNMENT_PATH, assignment_contract)
holdout_assignment["household_id"] = holdout_assignment["household_id"].astype("string")
holdout_assignment["holdout_group"] = holdout_assignment["holdout_group"].astype("string")
if holdout_assignment.household_id.duplicated().any():
    raise RuntimeError("Notebook 04B assignment has duplicate household IDs")
master["household_id"] = master["household_id"].astype("string")
master["target_date"] = pd.to_datetime(master["target_date"], errors="raise").dt.normalize()
if not master.row_id.is_unique: raise RuntimeError("Master split row_id must be unique")
full_model_dev = master.loc[master.internal_split.eq("model_development")].copy()
if full_model_dev.empty: raise RuntimeError("Frozen master split has no model_development rows")
MODEL_DEVELOPMENT_END = full_model_dev.target_date.max()
intentional_holdout_ids = set(holdout_assignment.loc[holdout_assignment.holdout_group.eq("intentional_holdout"), "household_id"].astype(str))
if not intentional_holdout_ids: raise RuntimeError("Notebook 04B intentional household holdout is empty")
if not set(master.household_id.astype(str)).issubset(set(holdout_assignment.household_id.astype(str))):
    raise RuntimeError("Notebook 04B assignment does not cover the frozen master split")
model_dev = full_model_dev.loc[~full_model_dev.household_id.astype(str).isin(intentional_holdout_ids)].copy()
if model_dev.empty: raise RuntimeError("No model-development rows remain after intentional holdout exclusion")
model_dev_ids = set(model_dev.row_id.astype(str))
if not model_dev.outer_split.eq("development").all():
    raise RuntimeError("model_development contains a row outside outer development")
if set(model_dev.household_id.astype(str)) & intentional_holdout_ids:
    raise RuntimeError("Intentional holdout households leaked into Notebook 05 selection rows")

loaded = {}
input_checks = []
for target in ["P95", "P90"]:
    path, contract_path = DATASETS[target], CONTRACTS[target]
    contract = read_contract(contract_path)
    signature_key = f"{target.lower()}_model_dataset"
    expected_signature = split_manifest["input_signatures"][signature_key]
    if sha256(path) != expected_signature["sha256"] or path.stat().st_size != expected_signature["size_bytes"]:
        raise RuntimeError(f"{target} model-ready dataset no longer matches the split-design input signature")
    features = FEATURES[target]
    required_columns = ["row_id", "household_id", "measurement_date", *features, "target"]
    absent = sorted(set(required_columns) - set(contract["columns"]))
    if absent: raise RuntimeError(f"{target} required features are absent from its data contract: {absent}")
    frame, selected_physical_rows = read_model_development_rows(path, contract, required_columns, model_dev_ids)
    frame["row_id"] = frame["row_id"].astype("string")
    frame["household_id"] = frame["household_id"].astype("string")
    frame["measurement_date"] = pd.to_datetime(frame["measurement_date"], errors="raise").dt.normalize()
    if frame.measurement_date.gt(MODEL_DEVELOPMENT_END).any():
        raise RuntimeError(f"{target} prefix unexpectedly includes a date beyond model development")
    if set(frame.row_id.astype(str)) != model_dev_ids:
        raise RuntimeError(f"{target} model-development IDs differ from the frozen master split")
    if not frame.target.isin([0, 1]).all(): raise RuntimeError(f"{target} labels are not binary")
    loaded[target] = frame.sort_values(["measurement_date", "household_id"], kind="mergesort").reset_index(drop=True)
    input_checks.append({"target": target, "passed": True, "model_development_rows": len(frame),
                         "positive_events": int(frame.target.sum()), "date_end": str(frame.measurement_date.max().date()),
                         "selected_physical_rows": selected_physical_rows})

p95, p90 = loaded["P95"], loaded["P90"]
if not p95[["row_id", "household_id", "measurement_date"]].astype(str).equals(p90[["row_id", "household_id", "measurement_date"]].astype(str)):
    raise RuntimeError("P95 and P90 model-development rows/order/keys differ")
input_checks_df = pd.DataFrame(input_checks)
input_checks_df.to_csv(TABLE_DIR / "05_01_model_development_input_checks.csv", index=False, encoding="utf-8")
display(input_checks_df)
print(f"Loaded non-held-out model-development rows only: {len(p95):,} rows from {p95.household_id.nunique():,} households; excluded {len(intentional_holdout_ids):,} intentional holdout households; date range {p95.measurement_date.min().date()} through {MODEL_DEVELOPMENT_END.date()}.")


## Stage 2 — build identical expanding folds and non-ML baselines

Fold boundaries use cumulative non-held-out model-development row counts and whole dates. The folds validate successively later periods: 55–70%, 70–85%, and 85–100%, while each training set expands from the beginning. Baselines use training-fold prevalence and the target-specific `lag1_peak_over_p95` / `lag1_peak_over_p90` ratio.


In [ ]:
def numeric_x(frame, features):
    return frame[features].apply(lambda column: pd.to_numeric(column, errors="coerce").astype("float64"))

daily_counts = p95.groupby("measurement_date", as_index=False).size().rename(columns={"size": "rows"}).sort_values("measurement_date")
daily_counts["cumulative_fraction"] = daily_counts.rows.cumsum() / daily_counts.rows.sum()
cutoffs = []
for fraction in FOLD_CUMULATIVE_ROW_FRACTIONS[:-1]:
    ix = int((daily_counts.cumulative_fraction - fraction).abs().idxmin())
    cutoffs.append(pd.Timestamp(daily_counts.loc[ix, "measurement_date"]).normalize())
cutoffs.append(MODEL_DEVELOPMENT_END)
if not all(left < right for left, right in zip(cutoffs, cutoffs[1:])):
    raise RuntimeError(f"CV row-fraction cutoffs are not strictly chronological: {cutoffs}")

folds=[]
previous_cutoff=None
for fold_number, (train_cutoff, validation_end) in enumerate(zip(cutoffs[:-1], cutoffs[1:]), start=1):
    train_mask=p95.measurement_date.le(train_cutoff)
    valid_mask=p95.measurement_date.gt(train_cutoff) & p95.measurement_date.le(validation_end)
    if previous_cutoff is not None:
        train_mask=p95.measurement_date.le(train_cutoff)
    fold={"fold":fold_number,"train_end":train_cutoff,"validation_start":p95.loc[valid_mask,"measurement_date"].min(),
          "validation_end":validation_end,"train_mask":train_mask,"validation_mask":valid_mask}
    if not train_mask.any() or not valid_mask.any(): raise RuntimeError(f"Fold {fold_number} has an empty train or validation block")
    folds.append(fold)
    previous_cutoff=train_cutoff

fold_rows=[]
for fold in folds:
    row={"fold":fold["fold"],"train_end":fold["train_end"],"validation_start":fold["validation_start"],
         "validation_end":fold["validation_end"],"train_rows":int(fold["train_mask"].sum()),
         "validation_rows":int(fold["validation_mask"].sum())}
    for target,frame in [("P95",p95),("P90",p90)]:
        y_train=frame.loc[fold["train_mask"],"target"].astype("int8")
        y_valid=frame.loc[fold["validation_mask"],"target"].astype("int8")
        if y_train.nunique()!=2 or y_valid.nunique()!=2:
            raise RuntimeError(f"{target} fold {fold['fold']} must have both classes in train and validation")
        row[f"{target}_train_events"]=int(y_train.sum())
        row[f"{target}_validation_events"]=int(y_valid.sum())
    fold_rows.append(row)
fold_summary=pd.DataFrame(fold_rows)
fold_summary.to_csv(TABLE_DIR / "05_02_time_cv_fold_design.csv",index=False,encoding="utf-8")
display(fold_summary)

baseline_rows=[]
for target,frame in [("P95",p95),("P90",p90)]:
    ratio_feature="lag1_peak_over_p95" if target=="P95" else "lag1_peak_over_p90"
    for fold in folds:
        train=frame.loc[fold["train_mask"]]
        valid=frame.loc[fold["validation_mask"]]
        y_train=train.target.astype("int8").to_numpy()
        y_valid=valid.target.astype("int8").to_numpy()
        prevalence=float(y_train.mean())
        heuristic_train=pd.to_numeric(train[ratio_feature],errors="coerce")
        heuristic_valid=pd.to_numeric(valid[ratio_feature],errors="coerce")
        median=float(heuristic_train.median()) if heuristic_train.notna().any() else 0.0
        heuristic_valid=heuristic_valid.fillna(median).to_numpy(dtype=float)
        for name,probability in [("Prevalence baseline",np.full(len(valid),prevalence)),
                                 ("Household-relative lag1 ratio",heuristic_valid)]:
            baseline_rows.append({"target":target,"model":name,"fold":fold["fold"],"train_rows":len(train),
                "validation_rows":len(valid),"train_prevalence":prevalence,
                "pr_auc":average_precision_score(y_valid,probability),"roc_auc":roc_auc_score(y_valid,probability),
                "brier_score":brier_score_loss(y_valid,probability) if name=="Prevalence baseline" else np.nan})
baseline_fold_metrics=pd.DataFrame(baseline_rows)
baseline_summary=baseline_fold_metrics.groupby(["target","model"],as_index=False).agg(
    mean_pr_auc=("pr_auc","mean"),std_pr_auc=("pr_auc","std"),mean_roc_auc=("roc_auc","mean"),mean_brier_score=("brier_score","mean"))
baseline_fold_metrics.to_csv(TABLE_DIR / "05_03_baseline_fold_metrics.csv",index=False,encoding="utf-8")
baseline_summary.to_csv(TABLE_DIR / "05_04_baseline_summary.csv",index=False,encoding="utf-8")
display(baseline_summary)


## Stage 3 — randomized hyperparameter search

Every candidate uses the same three folds for each target. Preprocessing is fitted inside each training fold. The randomized candidate budgets are intentionally explicit and are recorded in the manifest. XGBoost `scale_pos_weight` is tuned as a fold-specific multiple of the training-fold class ratio.


In [ ]:
LOGISTIC_GRID={"C":[0.01,0.1,1.0,10.0],"class_weight":[None,"balanced"]}
RF_GRID={
    "n_estimators":[100,200],"max_depth":[8,12,20],"min_samples_leaf":[5,10,20],
    "max_features":["sqrt",0.5,0.8],"class_weight":[None,"balanced","balanced_subsample"]}
XGB_GRID={
    "n_estimators":[100,200,300],"max_depth":[2,3,4],"learning_rate":[0.02,0.05,0.10],
    "min_child_weight":[1,5,10],"subsample":[0.7,0.85,1.0],"colsample_bytree":[0.7,0.85,1.0],
    "reg_alpha":[0,0.1,1],"reg_lambda":[1,5,10],"scale_pos_weight_mode":["one","sqrt_ratio","half_ratio","full_ratio"]}
CAT_GRID={"iterations":[100,200,300],"depth":[4,6,8],"learning_rate":[0.03,0.05,0.10],
          "l2_leaf_reg":[3,5,10],"auto_class_weights":[None,"Balanced"]}

candidate_configs={"Logistic Regression":[dict(C=C,class_weight=w) for C,w in itertools.product(LOGISTIC_GRID["C"],LOGISTIC_GRID["class_weight"])],
    "Random Forest":list(ParameterSampler(RF_GRID,n_iter=SEARCH_BUDGETS["Random Forest"],random_state=RANDOM_STATE)),
    "XGBoost":list(ParameterSampler(XGB_GRID,n_iter=SEARCH_BUDGETS["XGBoost"],random_state=RANDOM_STATE+1)),
    "CatBoost":list(ParameterSampler(CAT_GRID,n_iter=SEARCH_BUDGETS["CatBoost"],random_state=RANDOM_STATE+2))}

def stable_config_id(family,params):
    raw=json.dumps({"family":family,"params":params},sort_keys=True,default=str)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:12]

def build_estimator(family,params,train_y):
    if family=="Logistic Regression":
        return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),
            ("scaler",StandardScaler()),("model",LogisticRegression(C=float(params["C"]),penalty="l2",
             class_weight=params["class_weight"],solver="liblinear",max_iter=2000,random_state=RANDOM_STATE))])
    if family=="Random Forest":
        return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),
            ("model",RandomForestClassifier(n_estimators=int(params["n_estimators"]),max_depth=params["max_depth"],
             min_samples_leaf=int(params["min_samples_leaf"]),max_features=params["max_features"],
             class_weight=params["class_weight"],n_jobs=N_JOBS,random_state=RANDOM_STATE))])
    if family=="XGBoost":
        ratio=float((train_y==0).sum()/max((train_y==1).sum(),1))
        weight={"one":1.0,"sqrt_ratio":float(np.sqrt(ratio)),"half_ratio":ratio/2.0,"full_ratio":ratio}[params["scale_pos_weight_mode"]]
        return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),
            ("model",XGBClassifier(n_estimators=int(params["n_estimators"]),max_depth=int(params["max_depth"]),
             learning_rate=float(params["learning_rate"]),min_child_weight=int(params["min_child_weight"]),
             subsample=float(params["subsample"]),colsample_bytree=float(params["colsample_bytree"]),
             reg_alpha=float(params["reg_alpha"]),reg_lambda=float(params["reg_lambda"]),scale_pos_weight=weight,
             objective="binary:logistic",eval_metric="logloss",tree_method="hist",n_jobs=N_JOBS,
             random_state=RANDOM_STATE,verbosity=0))])
    if family=="CatBoost":
        return CatBoostClassifier(iterations=int(params["iterations"]),depth=int(params["depth"]),
            learning_rate=float(params["learning_rate"]),l2_leaf_reg=float(params["l2_leaf_reg"]),
            auto_class_weights=params["auto_class_weights"],loss_function="Logloss",eval_metric="Logloss",
            random_seed=RANDOM_STATE,verbose=False,allow_writing_files=False,thread_count=N_JOBS)
    raise KeyError(family)

def predict_probability(model,family,train_x,valid_x,train_y):
    if family=="CatBoost":
        model.fit(train_x,train_y,verbose=False)
        return model.predict_proba(valid_x)[:,1]
    model.fit(train_x,train_y)
    return model.predict_proba(valid_x)[:,1]

fold_metric_rows=[]
config_rows=[]
for target,frame in [("P95",p95),("P90",p90)]:
    features=FEATURES[target]
    X=numeric_x(frame,features)
    y=frame.target.astype("int8").to_numpy()
    for family,configs in candidate_configs.items():
        print(f"Tuning {target} {family}: {len(configs)} configurations × {len(folds)} folds")
        for config_number,params in enumerate(configs, start=1):
            config_id=stable_config_id(family,params)
            serialized_params=json.dumps(params,sort_keys=True,default=str)
            config_rows.append({"target":target,"model_family":family,"config_id":config_id,"parameters_json":serialized_params})
            for fold in folds:
                train_mask=fold["train_mask"].to_numpy()
                valid_mask=fold["validation_mask"].to_numpy()
                y_train,y_valid=y[train_mask],y[valid_mask]
                estimator=build_estimator(family,params,y_train)
                probability=predict_probability(estimator,family,X.loc[train_mask],X.loc[valid_mask],y_train)
                fold_metric_rows.append({"target":target,"model_family":family,"config_id":config_id,
                    "parameters_json":serialized_params,"fold":fold["fold"],"train_end":str(fold["train_end"].date()),
                    "validation_start":str(fold["validation_start"].date()),"validation_end":str(fold["validation_end"].date()),
                    "train_rows":int(train_mask.sum()),"validation_rows":int(valid_mask.sum()),
                    "train_events":int(y_train.sum()),"validation_events":int(y_valid.sum()),
                    "pr_auc":float(average_precision_score(y_valid,probability)),
                    "roc_auc":float(roc_auc_score(y_valid,probability)),
                    "brier_score":float(brier_score_loss(y_valid,probability))})
            if config_number % 5 == 0 or config_number == len(configs):
                print(f"  completed {config_number}/{len(configs)} configurations")

cv_fold_metrics=pd.DataFrame(fold_metric_rows)
config_table=pd.DataFrame(config_rows).drop_duplicates(["target","model_family","config_id"])
cv_fold_metrics.to_csv(TABLE_DIR / "05_05_cv_fold_metrics.csv",index=False,encoding="utf-8",float_format="%.17g")


## Stage 4 — summarize stability, select models, and save handoff

Within a model family, configurations within 0.005 mean PR-AUC of that family’s best are compared by lower fold variability and then simpler settings. Across families, the same PR-AUC tolerance is used; lower fold variability wins, with Logistic Regression preferred as the simpler tie-break. Baselines are reported but are not candidate classifier families.


In [ ]:
candidate_summary=cv_fold_metrics.groupby(["target","model_family","config_id","parameters_json"],as_index=False).agg(
    mean_pr_auc=("pr_auc","mean"),std_pr_auc=("pr_auc","std"),mean_roc_auc=("roc_auc","mean"),
    std_roc_auc=("roc_auc","std"),mean_brier_score=("brier_score","mean"),fold_count=("fold","nunique"))

def simplicity_key(family,params):
    if family=="Logistic Regression": return (float(params["C"]),)
    if family=="Random Forest": return (999 if params["max_depth"] is None else int(params["max_depth"]),-int(params["min_samples_leaf"]),int(params["n_estimators"]))
    if family=="XGBoost": return (int(params["max_depth"]),-float(params["reg_lambda"]),-float(params["reg_alpha"]),int(params["n_estimators"]))
    return (int(params["depth"]),-float(params["l2_leaf_reg"]),int(params["iterations"]))

def best_stable_config(frame):
    best_mean=frame.mean_pr_auc.max()
    tied=frame.loc[frame.mean_pr_auc.ge(best_mean-SELECTION_TOLERANCE_PR_AUC)].copy()
    tied["params"]=[json.loads(value) for value in tied.parameters_json]
    tied["simplicity"]=[simplicity_key(family,params) for family,params in zip(tied.model_family,tied.params)]
    return tied.sort_values(["std_pr_auc","simplicity","config_id"],ascending=[True,True,True],kind="mergesort").iloc[0]

family_winners=[]
for (target,family),group in candidate_summary.groupby(["target","model_family"],sort=False):
    chosen=best_stable_config(group)
    family_winners.append(chosen.to_dict())
family_winners=pd.DataFrame(family_winners)
family_order={"Logistic Regression":0,"Random Forest":1,"XGBoost":2,"CatBoost":3}
selected_rows=[]
for target,group in family_winners.groupby("target",sort=False):
    best_mean=group.mean_pr_auc.max()
    tied=group.loc[group.mean_pr_auc.ge(best_mean-SELECTION_TOLERANCE_PR_AUC)].copy()
    tied["family_order"]=tied.model_family.map(family_order)
    chosen=tied.sort_values(["std_pr_auc","family_order"],ascending=[True,True],kind="mergesort").iloc[0].to_dict()
    chosen["selection_rule"]="within 0.005 mean PR-AUC: lower fold SD, then simpler family/config"
    selected_rows.append(chosen)
selected_models=pd.DataFrame(selected_rows)

candidate_summary=candidate_summary.sort_values(["target","model_family","mean_pr_auc","std_pr_auc"],ascending=[True,True,False,True]).reset_index(drop=True)
family_winners.to_csv(TABLE_DIR / "05_06_best_config_per_family.csv",index=False,encoding="utf-8")
candidate_summary.to_csv(TABLE_DIR / "05_07_candidate_summary.csv",index=False,encoding="utf-8",float_format="%.17g")
selected_models.to_csv(TABLE_DIR / "05_08_selected_model_summary.csv",index=False,encoding="utf-8",float_format="%.17g")
display(selected_models[["target","model_family","mean_pr_auc","std_pr_auc","mean_roc_auc","mean_brier_score","parameters_json","selection_rule"]])

# Save the top ten configurations per target/family for review.
for (target,family),group in candidate_summary.groupby(["target","model_family"],sort=False):
    group.head(10).to_csv(TABLE_DIR / f"05_09_top10_{target.lower()}_{family.lower().replace(' ','_')}.csv",index=False,encoding="utf-8")

# Fold-performance chart uses each family’s selected configuration, not outer-test scores.
for target in ["P95","P90"]:
    target_winners=family_winners.loc[family_winners.target.eq(target)]
    plot_data=[]; labels=[]
    for _,winner in target_winners.iterrows():
        metrics=cv_fold_metrics.loc[(cv_fold_metrics.target.eq(target)) & cv_fold_metrics.model_family.eq(winner.model_family)
                                   & cv_fold_metrics.config_id.eq(winner.config_id),"pr_auc"].to_numpy()
        plot_data.append(metrics); labels.append(winner.model_family)
    fig,ax=plt.subplots(figsize=(9,5))
    ax.boxplot(plot_data,tick_labels=labels,showmeans=True)
    ax.set_title(f"{target} selected-configuration PR-AUC across expanding folds")
    ax.set_ylabel("Average precision (PR-AUC)")
    ax.grid(axis="y",alpha=0.25)
    plt.xticks(rotation=15,ha="right")
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f"05_{target.lower()}_selected_family_fold_pr_auc.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close(fig)

# XGBoost parameter-effect summaries show fold-mean PR-AUC by sampled parameter value.
xgb_configs=candidate_summary.loc[candidate_summary.model_family.eq("XGBoost")].copy()
xgb_configs["params"]=[json.loads(value) for value in xgb_configs.parameters_json]
for target in ["P95","P90"]:
    part=xgb_configs.loc[xgb_configs.target.eq(target)].copy()
    fig,axes=plt.subplots(1,3,figsize=(15,4.5))
    for ax,key,label in zip(axes,["max_depth","learning_rate","scale_pos_weight_mode"],["Max depth","Learning rate","Scale positive weight"]):
        part[key]=part.params.map(lambda params: params[key])
        effect=part.groupby(key,as_index=False,dropna=False).agg(mean_pr_auc=("mean_pr_auc","mean"),configs=("config_id","nunique"))
        ax.plot(effect[key].astype(str),effect.mean_pr_auc,marker="o")
        ax.set_title(label); ax.set_xlabel(label); ax.set_ylabel("Mean CV PR-AUC"); ax.grid(alpha=0.25)
    fig.suptitle(f"{target} XGBoost sampled parameter effects")
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f"05_{target.lower()}_xgboost_parameter_effects.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close(fig)

selected_payload={}
for _,row in selected_models.iterrows():
    selected_payload[row.target]={"model_family":row.model_family,"parameters":json.loads(row.parameters_json),
        "feature_list":FEATURES[row.target],"selection_metric":"mean PR-AUC","mean_pr_auc":float(row.mean_pr_auc),
        "std_pr_auc":float(row.std_pr_auc),"mean_roc_auc":float(row.mean_roc_auc),
        "mean_brier_score":float(row.mean_brier_score),"selection_rule":row.selection_rule}
selected_path=TABLE_DIR / "05_selected_models.json"
selected_path.write_text(json.dumps(selected_payload,indent=2,default=str),encoding="utf-8")

baseline_summary.to_csv(TABLE_DIR / "05_10_baseline_summary.csv",index=False,encoding="utf-8",float_format="%.17g")


## Stage 5 — final development-only validation

The final checks confirm each dataset contains only model-development IDs, all candidate configurations used all three chronological folds, selected features are target-specific, and no outer-test stage or test score was produced. The selection file is the handoff for calibration and threshold design in Notebook 06.


In [ ]:
expected_targets={"P95","P90"}
expected_families={"Logistic Regression","Random Forest","XGBoost","CatBoost"}
selected_json=json.loads(selected_path.read_text(encoding="utf-8"))
final_checks=pd.DataFrame([
    ("both_targets_loaded_from_nonholdout_model_development_only", set(loaded)==expected_targets and all(set(frame.row_id.astype(str))==model_dev_ids for frame in loaded.values()), f"{len(model_dev_ids):,} non-held-out frozen IDs"),
    ("intentional_holdout_excluded_from_all_selection_inputs", not (set(model_dev.household_id.astype(str)) & intentional_holdout_ids), f"{len(intentional_holdout_ids):,} assigned households excluded"),
    ("04b_assignment_manifest_verified", sha256(HOLDOUT_ASSIGNMENT_PATH)==holdout_manifest["assignment"]["sha256"] and bool(holdout_manifest.get("validation",{}).get("assignment_groups_are_disjoint_and_complete",False)), str(HOLDOUT_ASSIGNMENT_PATH)),
    ("model_development_dates_precede_outer_test", MODEL_DEVELOPMENT_END <= pd.Timestamp(split_manifest["selected_cutoff_inclusive_for_development"]), f"model_dev_end={MODEL_DEVELOPMENT_END.date()}"),
    ("three_whole_date_expanding_folds", len(folds)==3 and all(f["train_end"] < f["validation_end"] for f in folds), str(fold_summary[["fold","train_end","validation_start","validation_end"]].to_dict("records"))),
    ("each_candidate_uses_all_folds", cv_fold_metrics.groupby(["target","model_family","config_id"]).fold.nunique().eq(3).all(), f"{cv_fold_metrics.groupby(['target','model_family','config_id']).ngroups:,} configurations"),
    ("all_model_families_tuned_for_both_targets", set(candidate_summary.target)==expected_targets and set(candidate_summary.model_family)==expected_families, str(candidate_summary.groupby(["target","model_family"]).size().to_dict())),
    ("selected_model_handoff_contains_target_features", set(selected_json)==expected_targets and all(selected_json[target]["feature_list"]==FEATURES[target] for target in expected_targets), str({target:selected_json[target]["model_family"] for target in selected_json})),
    ("baselines_reported_for_both_targets", set(baseline_summary.target)==expected_targets and baseline_summary.model.nunique()==2, str(baseline_summary[["target","model","mean_pr_auc"]].to_dict("records"))),
    ("no_outer_test_metrics_created", not any(column in cv_fold_metrics.columns for column in ["test_seen_pr_auc","test_unseen_pr_auc"]) and not cv_fold_metrics.get("stage",pd.Series(dtype=str)).eq("test_seen").any(), "CV contains model-development folds only"),
    ("selection_and_cv_artifacts_exist", all(path.exists() and path.stat().st_size>0 for path in [selected_path,TABLE_DIR/"05_05_cv_fold_metrics.csv",TABLE_DIR/"05_07_candidate_summary.csv",TABLE_DIR/"05_08_selected_model_summary.csv"]), str(selected_path)),
    ("fold_and_parameter_figures_exist", all((FIGURE_DIR/f"05_{target.lower()}_selected_family_fold_pr_auc.png").exists() and (FIGURE_DIR/f"05_{target.lower()}_xgboost_parameter_effects.png").exists() for target in ["P95","P90"]), str(FIGURE_DIR)),
],columns=["check","passed","evidence"])
final_checks.to_csv(TABLE_DIR/"05_11_final_validation.csv",index=False,encoding="utf-8")
display(final_checks)
if not final_checks.passed.all(): raise RuntimeError(f"Model-tuning validation failed: {final_checks.loc[~final_checks.passed,'check'].tolist()}")

manifest={
    "created_utc":datetime.now(timezone.utc).isoformat(),"stage":"05_model_training_tuning",
    "input_signatures":{target:{"path":str(DATASETS[target]),"sha256":sha256(DATASETS[target]),"size_bytes":DATASETS[target].stat().st_size} for target in ["P95","P90"]},
    "master_split_path":str(SPLIT_PATH),"master_split_sha256":sha256(SPLIT_PATH),
    "training_scope":"internal_split == model_development excluding all 04B intentional_holdout households","outer_test_labels_loaded":False,"outer_test_metrics_created":False,
    "intentional_holdout_assignment_path":str(HOLDOUT_ASSIGNMENT_PATH),"intentional_holdout_assignment_sha256":sha256(HOLDOUT_ASSIGNMENT_PATH),
    "intentional_holdout_households_excluded_from_model_selection":True,"intentional_holdout_household_count":len(intentional_holdout_ids),
    "nonholdout_model_development_rows":len(model_dev),"full_model_development_rows":len(full_model_dev),
    "model_development_rows":len(model_dev),"model_development_end":str(MODEL_DEVELOPMENT_END.date()),
    "folds":fold_summary.to_dict("records"),"primary_metric":"mean PR-AUC","secondary_metrics":["ROC-AUC","Brier score","fold-to-fold PR-AUC SD"],
    "selection_tolerance_pr_auc":SELECTION_TOLERANCE_PR_AUC,"search_budgets":SEARCH_BUDGETS,
    "candidate_configurations_per_family":{family:len(configs) for family,configs in candidate_configs.items()},
    "selected_models_path":str(selected_path),"selected_models":selected_payload,
    "baseline_summary_path":str(TABLE_DIR/"05_10_baseline_summary.csv"),
    "validation":{str(row.check):bool(row.passed) for row in final_checks.itertuples(index=False)},
    "software":{"python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__},
}
(TABLE_DIR/"05_model_training_tuning_manifest.json").write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8")
print("Model-family tuning completed; all checks passed.")
print("Selected handoff:",selected_path)
print("No outer-test metrics were calculated.")


## Interpretation and handoff

The selected families and hyperparameters come from time-aware cross-validation that excludes every 04B intentional-holdout household. Notebook 06 may fit the primary temporal-evaluation model using its prescribed development split; Notebook 10 separately refits those frozen settings without intentional-holdout labels for the household-transfer experiment. The outer test remains reserved for final evaluation.
